In [ ]:
# CELL 1: Install dependencies
!pip install -q langchain langchain-openai langchain-community langchain-chroma
!pip install -q langchain-huggingface sentence-transformers
!pip install -q tavily-python
!pip install -q networkx scikit-learn
!pip install -q chromadb


In [ ]:
# CELL 2: Mount Drive, Imports, API Keys

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import re
import json
import csv
import torch
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any
from datetime import datetime
import concurrent.futures

import chromadb
from chromadb.utils import embedding_functions

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from tavily import TavilyClient

# ── API Keys ──────────────────────────────────────────────────────────────────
import os

os.environ["OPENAI_API_KEY"] = os.environ["SARVAM_API_KEY"]
# TAVILY_API_KEY is read from the environment directly.
os.environ["LANGCHAIN_TRACING_V2"] = "false"

print("✅ Drive mounted, imports done, API keys set.")


In [ ]:
# Temp test cell
import os

path = "/content/drive/MyDrive/Hindi_CN_Research/chroma_db"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("Contents:")
    print(os.listdir(path)[:20])

In [ ]:
# CELL 3: LLM + ChromaDB

SEP = "=" * 80

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="sarvam-105b",
    temperature=0,
    base_url="https://api.sarvam.ai/v1",
    api_key=os.environ["OPENAI_API_KEY"],
    timeout=45,
    max_retries=1,
)
print("✅ LLM ready.")

# ── ChromaDB ──────────────────────────────────────────────────────────────────
CHROMA_PATH     = "/content/drive/MyDrive/Hindi_CN_Research/chroma_db"
COLLECTION_NAME = "hindi_cn_db"

device_type = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   Device: {device_type}")

chroma_available  = False
chroma_collection = None

try:
    chroma_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="paraphrase-multilingual-MiniLM-L12-v2",
        device=device_type,
    )
    chroma_client     = chromadb.PersistentClient(path=CHROMA_PATH)
    chroma_collection = chroma_client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=chroma_ef,
    )
    doc_count = chroma_collection.count()
    print(f"✅ ChromaDB ready | Collection: '{COLLECTION_NAME}' | Docs: {doc_count:,}")
    chroma_available = True if doc_count > 0 else False

except Exception as e:
    print(f"❌ ChromaDB failed: {e}")
    print("   Tavily-only mode active.")

print(f"\n✅ Cell 3 complete.")
print(f"   LLM      : ready")
print(f"   ChromaDB : {'ready (' + str(chroma_collection.count()) + ' docs)' if chroma_available else 'unavailable'}")


In [ ]:
# CELL 4: Helper functions

# ─────────────────────────────────────────────────────────────────────────────
# HELPER 1: Clean web text — remove HTML, markdown links, URLs, boilerplate
# ─────────────────────────────────────────────────────────────────────────────
def clean_web_text(text: str) -> str:
    """
    Strips HTML tags, markdown links, bare URLs, duplicate lines,
    and navigation boilerplate from Tavily web content.
    Applied BEFORE LexRank so LexRank works on clean sentences only.
    """
    if not text:
        return ""

    # Remove markdown image tags: ![alt](url)
    text = re.sub(r'!\[[^\]]*\]\([^)]+\)', '', text)

    # Remove markdown links: [text](url) → keep text only
    text = re.sub(r'\[([^\]]*)\]\([^)]+\)', r'\1', text)

    # Remove bare URLs
    text = re.sub(r'https?://\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove lines with mostly special characters / navigation artifacts
    text = re.sub(r'[›»«‹\*\|\-]{2,}', ' ', text)

    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Split into lines, filter short/duplicate lines
    lines = text.split('\n')
    seen       = set()
    clean_lines = []
    for line in lines:
        line = line.strip()
        # Keep only lines with real content (>40 chars)
        if len(line) < 40:
            continue
        # Deduplicate
        normalized = re.sub(r'\s+', ' ', line.lower())
        if normalized in seen:
            continue
        seen.add(normalized)
        clean_lines.append(line)

    return '\n'.join(clean_lines)


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 2: LexRank — reduce long article to top N sentences
# ─────────────────────────────────────────────────────────────────────────────
def lexrank_extract(text: str, num_sentences: int = 12) -> str:
    """
    Applies LexRank to select the top `num_sentences` from `text`.
    Splits on Hindi Purna Viram (।) and standard punctuation.
    Returns sentences in original chronological order.
    This is NOT the final summary — only a pre-filter before LLM summarization.
    """
    sentences = [
        s.strip()
        for s in re.split(r'[।?!\.]', text)
        if len(s.strip()) > 15
    ]

    if not sentences:
        return text

    if len(sentences) <= num_sentences:
        return text  # already short enough

    try:
        vectorizer   = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except Exception as e:
        print(f"   [LexRank] TF-IDF failed ({e}), returning cleaned text.")
        return text

    sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
    nx_graph   = nx.from_numpy_array(sim_matrix)

    try:
        scores = nx.pagerank(nx_graph, max_iter=200)
    except nx.PowerIterationFailedConvergence:
        scores = nx.degree_centrality(nx_graph)

    top_idx = sorted(
        range(len(sentences)),
        key=lambda i: scores[i],
        reverse=True
    )[:num_sentences]

    top_idx_ordered = sorted(top_idx)  # restore chronological order
    return " । ".join([sentences[i] for i in top_idx_ordered]) + " ।"


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 3: Safe LLM call — no threads, uses LLM-level timeout
# ─────────────────────────────────────────────────────────────────────────────
def _safe_call(chain, inputs: dict, fallback: str = "") -> str:
    """
    Calls a LangChain chain safely.
    Timeout is handled by the httpx client inside ChatOpenAI (timeout=45).
    Returns fallback string on any error — never raises.
    No ThreadPoolExecutor — safe for Colab.
    """
    try:
        result = chain.invoke(inputs)
        return result if isinstance(result, str) else str(result)
    except Exception as e:
        print(f"   [LLM ❌] {type(e).__name__}: {str(e)[:100]}")
        return fallback


print("✅ Helper functions defined: clean_web_text(), lexrank_extract(), _safe_call()")


In [ ]:
# CELL 5: All pipeline functions
# All original LLM prompts preserved EXACTLY — do not modify them.

# ─────────────────────────────────────────────────────────────────────────────
# HELPER 1: Safe LLM call
# ─────────────────────────────────────────────────────────────────────────────
def _safe_call(chain, inputs: dict, fallback: str = "") -> str:
    try:
        result = chain.invoke(inputs)
        return result if isinstance(result, str) else str(result)
    except Exception as e:
        print(f"   [LLM ❌] {type(e).__name__}: {str(e)[:100]}")
        return fallback


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 2: Retry LLM call
# ─────────────────────────────────────────────────────────────────────────────
def _safe_call_with_retry(chain, inputs: dict, label: str, max_retries: int = 3) -> str:
    for attempt in range(1, max_retries + 1):
        try:
            result = chain.invoke(inputs)
            result = result.strip() if isinstance(result, str) else str(result).strip()
            if result:
                print(f"   [{label}] ✅ Generated on attempt {attempt}.")
                return result
            else:
                print(f"   [{label}] Attempt {attempt}: empty response.")
        except Exception as e:
            print(f"   [{label}] Attempt {attempt} failed: {type(e).__name__}: {str(e)[:80]}")

        wait = 3 * (2 ** (attempt - 1))
        print(f"   [{label}] Waiting {wait}s before retry...")
        time.sleep(wait)

    print(f"   [{label}] ⚠️  All {max_retries} retries exhausted.")
    return ""


# ─────────────────────────────────────────────────────────────────────────────
# HELPER 3: Detect readable Hindi content
# NEW — prevents garbled PDFs and encoded text from entering the pipeline
# ─────────────────────────────────────────────────────────────────────────────
def is_readable_hindi(text: str, min_devanagari_ratio: float = 0.1) -> bool:
    """
    Returns True if text contains enough Devanagari characters to be useful.
    Rejects:
      - Garbled PDFs (Romanized transliterations like 'ek&pk&[kcc')
      - Mostly-ASCII content returned for Hindi queries
      - Empty or very short text

    Devanagari Unicode range: U+0900 to U+097F
    Threshold of 10% is intentionally low to also allow mixed Hindi-English
    articles (like the APA.org women leadership article) to pass through.
    Pure garbage like 'ek&pk&[kcc' has 0% Devanagari so it is always rejected.
    """
    if not text or len(text) < 50:
        return False
    devanagari_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    ratio = devanagari_chars / len(text)
    return ratio >= min_devanagari_ratio


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 1: generate_queries
# ─────────────────────────────────────────────────────────────────────────────
def generate_queries(hate_speech: str) -> List[str]:
    # ORIGINAL PROMPT — DO NOT MODIFY
    template = """You are responding with counterspeech to the hate speech in Hindi:
{question}

Generate exactly THREE search queries in HINDI (Devanagari script) to retrieve facts, statistics, or constructive perspectives to counter this claim.

Rules:
- One query per line.
- No numbering.
- No bullets.
- No explanation.
- Keep every query under 200 characters."""

    chain    = ChatPromptTemplate.from_template(template) | llm | StrOutputParser()
    response = _safe_call(chain, {"question": hate_speech}, fallback="")

    print(SEP)
    print("STAGE 1 — QUERY GENERATION")
    print(SEP)
    print(f"Hate Speech:\n  {hate_speech}\n")
    print(f"Raw LLM Response:\n  {repr(response)}\n")

    queries = [
        q.strip("-•1234567890. \n")
        for q in response.split("\n")
        if q.strip()
    ]
    queries = [q[:200] for q in queries[:3]]

    print("Generated Hindi Queries:")
    for i, q in enumerate(queries, 1):
        print(f"  Q{i}: {q}")
    print(SEP)

    return queries


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 2: retrieve_documents
# ─────────────────────────────────────────────────────────────────────────────
def retrieve_documents(hate_speech: str, queries: List[str]) -> List[Dict]:
    """
    For each query:
      - 1 result from Tavily (web, min 150 chars, must pass Hindi readability check)
      - 1 result from ChromaDB (static, min 100 chars)
    Garbled PDFs and encoded text are detected and skipped automatically.
    """
    tavily_client = TavilyClient()
    all_docs      = []

    print(SEP)
    print("STAGE 2 — DOCUMENT RETRIEVAL")
    print(f"Strategy : 1 Tavily (web) + 1 ChromaDB (static) per query")
    print(f"Filters  : Tavily min 150 chars + Hindi readability check | ChromaDB min 100 chars")
    print(f"Target   : {len(queries) * 2} docs  |  "
          f"ChromaDB: {'available' if chroma_available else 'NOT AVAILABLE — Tavily only'}")
    print(SEP)

    for qi, query in enumerate(queries, 1):
        print(f"\n--- Query {qi}/{len(queries)}: {query} ---")

        # ── 1 from Tavily ─────────────────────────────────────────────────
        try:
            resp           = tavily_client.search(query=query, max_results=1)
            tavily_results = resp.get("results", [])
        except Exception as e:
            print(f"  [Tavily ❌] {e}")
            tavily_results = []

        if tavily_results:
            r          = tavily_results[0]
            raw_text   = r.get("content", "")
            clean_text = clean_web_text(raw_text)

            if len(clean_text) < 100:
                clean_text = raw_text

            # Check 1: minimum length
            if len(clean_text) < 150:
                print(f"  [Tavily  ⚠️ ] Too short ({len(clean_text)} chars) — skipping.")
                print(f"               Title: {r.get('title', '')[:70]}")

            # Check 2: readable Hindi / not garbled PDF
            elif not is_readable_hindi(clean_text):
                print(f"  [Tavily  ⚠️ ] Non-Hindi or garbled content detected — skipping.")
                print(f"               Title    : {r.get('title', '')[:70]}")
                print(f"               URL      : {r.get('url', '')}")
                devanagari = sum(1 for c in clean_text if '\u0900' <= c <= '\u097F')
                print(f"               Devanagari chars: {devanagari} / {len(clean_text)} ({devanagari/len(clean_text)*100:.1f}%)")

            else:
                all_docs.append({
                    "query":  query,
                    "text":   clean_text,
                    "title":  r.get("title", "No Title"),
                    "url":    r.get("url", ""),
                    "source": "tavily_web",
                })
                print(f"  [Tavily  ✅] {all_docs[-1]['title'][:70]}")
                print(f"               URL            : {all_docs[-1]['url']}")
                print(f"               Length (clean) : {len(clean_text)} chars")
        else:
            print(f"  [Tavily  ❌] No results.")

        # ── 1 from ChromaDB ───────────────────────────────────────────────
        if chroma_available and chroma_collection is not None:
            try:
                result     = chroma_collection.query(
                    query_texts=[query],
                    n_results=1,
                    include=["documents", "metadatas", "distances"],
                )
                docs_list  = result.get("documents",  [[]])[0]
                metas_list = result.get("metadatas",  [[]])[0]
                dist_list  = result.get("distances",  [[]])[0]

                if docs_list:
                    doc_text = str(docs_list[0]).strip()
                    meta     = metas_list[0] if metas_list else {}
                    dist     = dist_list[0]  if dist_list  else "N/A"

                    if len(doc_text) < 100:
                        print(f"  [ChromaDB ⚠️ ] Too short ({len(doc_text)} chars) — skipping.")
                    else:
                        title = (meta.get("source_csv")
                                 or meta.get("title")
                                 or meta.get("source")
                                 or "ChromaDB Document")
                        url   = meta.get("url", "")

                        all_docs.append({
                            "query":  query,
                            "text":   doc_text,
                            "title":  title,
                            "url":    url,
                            "source": "chromadb",
                        })
                        print(f"  [ChromaDB ✅] {title[:70]}")
                        print(f"                URL     : {url if url else '(no url in metadata)'}")
                        print(f"                Length  : {len(doc_text)} chars")
                        print(f"                Distance: {dist:.4f}" if isinstance(dist, float) else f"                Distance: {dist}")
                else:
                    print(f"  [ChromaDB ⚠️ ] No documents returned.")

            except Exception as e:
                print(f"  [ChromaDB ❌] {e}")
        else:
            print(f"  [ChromaDB ⏭️ ] Not available.")

    tavily_count = sum(1 for d in all_docs if d["source"] == "tavily_web")
    chroma_count = sum(1 for d in all_docs if d["source"] == "chromadb")
    print(f"\nRetrieval Summary:")
    print(f"  Tavily (web)     : {tavily_count} docs")
    print(f"  ChromaDB (static): {chroma_count} docs")
    print(f"  Total            : {len(all_docs)} docs")
    print(SEP)

    return all_docs


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 3: summarize_documents
# ─────────────────────────────────────────────────────────────────────────────
def summarize_documents(hate_speech: str, docs: List[Dict]) -> List[Dict]:
    # ORIGINAL PROMPT — DO NOT MODIFY
    template = """Summarize this document in 2-3 sentences.
The summary should:
- Answer the query.
- Preserve facts useful for countering the hate speech.
- Keep important statistics.
- Keep source information if mentioned.

Query:
{query}

Hate Speech:
{hs}

Document:
{document}"""

    chain = ChatPromptTemplate.from_template(template) | llm | StrOutputParser()

    print(SEP)
    print("STAGE 3 — LEXRANK EXTRACTION + LLM SUMMARIZATION")
    print(SEP)

    summarized_docs = []

    for i, doc in enumerate(docs, 1):
        print(f"\n--- Doc {i}/{len(docs)} | [{doc['source']}] {doc['title'][:60]} ---")
        print(f"  Original length : {len(doc['text'])} chars")

        lexrank_text = lexrank_extract(doc["text"], num_sentences=12)
        status = "reduced" if len(lexrank_text) < len(doc["text"]) else "unchanged — already short"
        print(f"  LexRank length  : {len(lexrank_text)} chars  ({status})")
        print(f"  LexRank preview : {lexrank_text[:200]}{'...' if len(lexrank_text) > 200 else ''}")

        summary = _safe_call(
            chain,
            {"query": doc["query"], "document": lexrank_text, "hs": hate_speech},
            fallback=lexrank_text[:400],
        ).strip()

        if not summary:
            summary = lexrank_text[:400]

        print(f"  LLM Summary     : {summary[:200]}{'...' if len(summary) > 200 else ''}")
        print(f"  URL             : {doc['url'] if doc['url'] else '(none)'}")

        summarized_docs.append({**doc, "lexrank_text": lexrank_text, "summary": summary})

    print(f"\nTotal summaries: {len(summarized_docs)}")
    print(SEP)

    return summarized_docs


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 4: critique_documents
# ─────────────────────────────────────────────────────────────────────────────
def critique_documents(hate_speech: str, summarized_docs: List[Dict]) -> List[Dict]:
    # ORIGINAL PROMPT — DO NOT MODIFY
    template = """You are a grader assessing whether a retrieved document is useful for generating counterspeech.

Retrieved document:
{document}

Hate speech:
{hs}

Search query:
{question}

Respond ONLY with:
yes
or
no

If the document helps counter the hate speech -> yes
If it is irrelevant or supports the hate speech -> no"""

    chain   = ChatPromptTemplate.from_template(template) | llm | StrOutputParser()
    POSITIVE = {"yes", "हाँ", "हां", "हा", "हाँ।", "हां।", "हाँ,", "हां,"}

    print(SEP)
    print("STAGE 4 — CRITIQUE / RELEVANCE GRADING")
    print(SEP)

    relevant_docs = []

    for i, doc in enumerate(summarized_docs, 1):
        print(f"\n--- Doc {i}/{len(summarized_docs)} | [{doc['source']}] {doc['title'][:60]} ---")
        print(f"  Summary preview : {doc['summary'][:120]}...")

        raw = _safe_call(
            chain,
            {"document": doc["summary"], "hs": hate_speech, "question": doc["query"]},
            fallback="no",
        ).strip()

        is_relevant = any(raw.lower().startswith(p.lower()) for p in POSITIVE)

        print(f"  Raw LLM response: '{raw}'")
        print(f"  Decision        : {'✅ RELEVANT — kept' if is_relevant else '❌ IRRELEVANT — discarded'}")

        if is_relevant:
            relevant_docs.append({**doc, "critique": "yes"})

    print(f"\nKept after critique: {len(relevant_docs)} / {len(summarized_docs)}")
    print(SEP)

    return relevant_docs


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 5: generate_counter_narratives
# ─────────────────────────────────────────────────────────────────────────────
def generate_counter_narratives(hate_speech: str, relevant_docs: List[Dict]) -> Dict[str, str]:
    # ALL 3 PROMPTS PRESERVED EXACTLY — DO NOT MODIFY

    def format_documents(docs: List[Dict]) -> str:
        parts = []
        for i, d in enumerate(docs, 1):
            entry = f"{i}. {d['summary']}"
            if d.get("title") and d.get("url"):
                entry += f"\n   source: {{title: '{d['title']}', url: '{d['url']}'}}"
            elif d.get("title"):
                entry += f"\n   source: {d['title']}"
            parts.append(entry)
        return "\n\n".join(parts)

    print(SEP)
    print("STAGE 5 — COUNTER-NARRATIVE GENERATION")
    print(SEP)

    if not relevant_docs:
        print("⚠️  No relevant documents — generating without factual context.")
        context_str = "कोई प्रासंगिक स्रोत उपलब्ध नहीं है।"
    else:
        context_str = format_documents(relevant_docs)
        print(f"Using {len(relevant_docs)} relevant document(s).")
        print(f"Context preview:\n{context_str[:400]}...\n")

    # ── PROMPT 1: FACT-BASED — DO NOT MODIFY ─────────────────────────────────
    fact_template = """You are an expert counterspeech generator tasked with mitigating online hate speech in Hindi using FACTUAL REFUTATION.

DEFINITION:
A Fact-based Counter-Narrative provides factual evidence, statistics, and reliable data from the context to correct misconceptions, refute false information, and challenge the accuracy or credibility of the hate speech.

HATE SPEECH:
"{question}"

RETRIEVED FACTUAL CONTEXT:
{context}

INSTRUCTIONS:
1. Write a Fact-based counter-narrative strictly in HINDI (Devanagari script).
2. Utilize the retrieved facts and statistics directly to correct false claims. If specific facts are unavailable in the context, focus on general factual and constitutional principles of democracy, election integrity, or civility.
3. Keep it concise (2-4 sentences) and maintain a neutral, objective, non-violent tone.
4. Do not include any intro, labels, or markdown formatting. Return ONLY the Hindi text."""

    cn_fact = _safe_call_with_retry(
        ChatPromptTemplate.from_template(fact_template) | llm | StrOutputParser(),
        {"context": context_str, "question": hate_speech},
        label="FACT-BASED",
    )
    print(f"--- Type 1: FACT-BASED ---")
    print(cn_fact if cn_fact else "⚠️  Empty after all retries.")
    print()
    time.sleep(5)   # increased from 3s to 5s to reduce rate-limit risk

    # ── PROMPT 2: POSITIVE RESPONSE — DO NOT MODIFY ──────────────────────────
    positive_template = """You are an expert counterspeech generator tasked with mitigating online hate speech in Hindi using POSITIVE DE-ESCALATION.

DEFINITION:
A Positive Response Counter-Narrative presents an optimistic, constructive, or supportive viewpoint to promote understanding, empathy, national unity, and harmony.

HATE SPEECH:
"{question}"

INSTRUCTIONS:
1. Write a Positive Response counter-narrative strictly in HINDI (Devanagari script).
2. Focus on shared values, empathy, sportsmanship/civility, and unity without engaging in debate.
3. Keep it concise (2-4 sentences) and de-escalate hostility gently.
4. Do not include any intro, labels, or markdown formatting. Return ONLY the Hindi text."""

    cn_positive = _safe_call_with_retry(
        ChatPromptTemplate.from_template(positive_template) | llm | StrOutputParser(),
        {"question": hate_speech},
        label="POSITIVE",
    )
    print(f"--- Type 2: POSITIVE RESPONSE ---")
    print(cn_positive if cn_positive else "⚠️  Empty after all retries.")
    print()
    time.sleep(5)   # increased from 3s to 5s

    # ── PROMPT 3: COUNTER QUESTION — DO NOT MODIFY ───────────────────────────
    question_template = """You are an expert counterspeech generator tasked with mitigating online hate speech in Hindi using REFRAMING QUESTIONS.

DEFINITION:
A Counter Question Counter-Narrative presents thought-provoking questions that challenge or probe underlying assumptions, biases, or logical inconsistencies to stimulate critical thinking and self-reflection.

HATE SPEECH:
"{question}"

RETRIEVED FACTUAL CONTEXT:
{context}

INSTRUCTIONS:
1. Write a Counter Question counter-narrative strictly in HINDI (Devanagari script).
2. Ask 1-2 sharp, respectful, and thought-provoking questions that challenge the speaker's logic or bias.
3. Keep it concise (2-4 sentences) and avoid sounding sarcastic or abusive.
4. Do not include any intro, labels, or markdown formatting. Return ONLY the Hindi text."""

    cn_question = _safe_call_with_retry(
        ChatPromptTemplate.from_template(question_template) | llm | StrOutputParser(),
        {"context": context_str, "question": hate_speech},
        label="COUNTER-Q",
    )
    print(f"--- Type 3: COUNTER QUESTION ---")
    print(cn_question if cn_question else "⚠️  Empty after all retries.")
    print(SEP)

    return {
        "Fact-based":        cn_fact,
        "Positive Response": cn_positive,
        "Counter Question":  cn_question,
    }


print("✅ All pipeline functions defined.")


In [ ]:
# CELL 6: run_pipeline() — fixed duplicate-key edge case in critique_results

def run_pipeline(hate_speech: str) -> Dict[str, Any]:
    """
    Runs the complete pipeline for a single Hindi hate-speech input.
    Prints detailed output at every stage.
    Returns structured dict with all intermediate + final outputs.

    critique_results correctly reflects actual LLM yes/no decisions,
    including cases where two ChromaDB docs share the same title and have no URL.
    """
    print(f"\n{'█'*80}")
    print(f"  HINDI COUNTER-NARRATIVE PIPELINE — SINGLE EXAMPLE")
    print(f"{'█'*80}\n")
    print(f"Input: {hate_speech}\n")

    # Stage 1
    queries = generate_queries(hate_speech)

    # Stage 2
    docs = retrieve_documents(hate_speech, queries)

    # Stage 3
    summarized_docs = summarize_documents(hate_speech, docs)

    # Stage 4
    relevant_docs = critique_documents(hate_speech, summarized_docs)

    # Stage 5
    counter_narratives = generate_counter_narratives(hate_speech, relevant_docs)

    source_urls = list({d["url"] for d in relevant_docs if d.get("url")})

    # ── Final summary print ───────────────────────────────────────────────────
    print(f"{'█'*80}")
    print(f"  PIPELINE COMPLETE — FINAL SUMMARY")
    print(f"{'█'*80}\n")

    print(f"📌 Hate Speech:\n   {hate_speech}\n")

    print(f"🔍 Queries:")
    for i, q in enumerate(queries, 1):
        print(f"   Q{i}: {q}")

    print(f"\n📄 Retrieved ({len(docs)} docs):")
    for i, d in enumerate(docs, 1):
        print(f"   {i}. [{d['source']:10}] {d['title'][:65]}")
        print(f"          URL: {d['url'] if d['url'] else '(no url)'}")

    print(f"\n✅ After Critique ({len(relevant_docs)}/{len(summarized_docs)} kept):")
    for i, d in enumerate(relevant_docs, 1):
        print(f"   {i}. [{d['source']:10}] {d['title'][:65]}")

    print(f"\n🗣️  Counter-Narratives:")
    for cn_type, cn_text in counter_narratives.items():
        print(f"\n  [{cn_type}]")
        print(f"   {cn_text if cn_text else '⚠️  Empty'}")

    print(f"\n🔗 Sources Used:")
    for url in source_urls:
        print(f"   {url}")

    print(f"\n{'█'*80}\n")

    # ── Build lookup set from docs that passed critique ───────────────────────
    # Uses title + url + first 50 chars of summary as unique key.
    # This handles the edge case where two ChromaDB docs share the same title
    # and have no URL (e.g. two rows from hindi_news_articles.csv) — without
    # the summary suffix, only one would be marked "yes" in critique_results.
    relevant_keys = {
        d["title"] + "||" + d.get("url", "") + "||" + d["summary"][:50]
        for d in relevant_docs
    }

    return {
        "hate_speech": hate_speech,
        "queries":     queries,

        "retrieved_documents": [
            {
                "title":       d["title"],
                "url":         d["url"],
                "source":      d["source"],
                "text_length": len(d["text"]),
            }
            for d in docs
        ],

        "lexrank_sentences": [
            {
                "title":        d["title"],
                "source":       d["source"],
                "lexrank_text": d["lexrank_text"],
            }
            for d in summarized_docs
        ],

        "summaries": [
            {
                "title":   d["title"],
                "source":  d["source"],
                "url":     d["url"],
                "summary": d["summary"],
            }
            for d in summarized_docs
        ],

        # critique value derived from relevant_keys lookup:
        # "yes" = LLM approved this doc at runtime
        # "no"  = LLM discarded this doc at runtime
        "critique_results": [
            {
                "title":    d["title"],
                "source":   d["source"],
                "url":      d["url"],
                "summary":  d["summary"],
                "critique": "yes" if (
                    d["title"] + "||" + d.get("url", "") + "||" + d["summary"][:50]
                ) in relevant_keys else "no",
            }
            for d in summarized_docs
        ],

        "selected_documents": [
            {
                "title":   d["title"],
                "source":  d["source"],
                "url":     d["url"],
                "summary": d["summary"],
            }
            for d in relevant_docs
        ],

        "counter_narratives": counter_narratives,
        "source_urls":        source_urls,
    }


print("✅ run_pipeline() defined.")


In [ ]:
import time

In [ ]:
# TEST 7 — Safely inspect the EXISTING Chroma collection
# Does NOT mount Drive
# Does NOT create a new database
# Does NOT add/delete anything

print("=" * 70)
print("CHROMADB CONNECTION TEST")
print("=" * 70)

try:
    print("Collection name :", chroma_collection.name)
    print("Document count  :", f"{chroma_collection.count():,}")

    print("\nTesting a tiny read/query...")

    result = chroma_collection.peek(limit=1)

    print("✅ ChromaDB responded successfully.")
    print("Keys returned   :", list(result.keys()))

    if result.get("ids"):
        print("Sample ID       :", result["ids"][0])

    if result.get("documents"):
        doc = result["documents"][0]
        print("Sample document :", str(doc)[:300])

    if result.get("metadatas"):
        print("Sample metadata :", result["metadatas"][0])

except Exception as e:
    print("❌ ChromaDB query FAILED")
    print("Error type :", type(e).__name__)
    print("Error     :", str(e))

print("=" * 70)

In [ ]:
import pandas as pd

INPUT_CSV = "/content/drive/MyDrive/Hindi_CN_Research/final_cn_output.csv"

df = pd.read_csv(INPUT_CSV)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head(3))

In [ ]:
#if any of 3 is there - skip that

import os
import time
import contextlib
import io
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "/content/drive/MyDrive/Hindi_CN_Research/final_cn_output.csv"

# Pause between pipeline runs.
SLEEP_BETWEEN_ROWS = 2


# ============================================================
# LOAD CURRENT CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

required_columns = [
    "text",
    "factual_cn",
    "counter_question_cn",
    "positive_cn",
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print(f"Loaded {len(df)} rows.")
print("Columns:", df.columns.tolist())


# ============================================================
# QUIET VERSION OF EXISTING PIPELINE
# ============================================================

def run_pipeline_quiet(hate_speech):
    """
    Runs the existing pipeline while hiding its intermediate
    stdout output from the notebook.
    """
    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        result = run_pipeline(hate_speech)

    return result


# ============================================================
# HELPER: SAFE NON-EMPTY CHECK
# ============================================================

def is_nonempty(value):
    """
    Correctly handles both empty strings and pandas NaN values.
    """
    return pd.notna(value) and str(value).strip() != ""


# ============================================================
# HELPER: CHECK IF ROW HAS ANY DATA (SKIP IF NOT COMPLETELY BLANK)
# ============================================================

def row_has_any_cn(row):
    """
    Returns True if ANY of the three CNs are already present.
    If True, the row will be skipped to prioritize completely untouched rows.
    """
    return (
        is_nonempty(row["factual_cn"])
        or is_nonempty(row["counter_question_cn"])
        or is_nonempty(row["positive_cn"])
    )

def row_is_complete(row):
    return (
        is_nonempty(row["factual_cn"])
        and is_nonempty(row["counter_question_cn"])
        and is_nonempty(row["positive_cn"])
    )


# ============================================================
# HELPER: SAFE CSV SAVE
# ============================================================

def save_progress():
    """
    Save to a temporary file first, then replace the original.
    This reduces the chance of leaving behind a partially-written CSV.
    """
    temp_path = CSV_PATH + ".tmp"

    df.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(temp_path, CSV_PATH)


# ============================================================
# PRE-RUN STATUS
# ============================================================

total_rows = len(df)

has_data = df.apply(row_has_any_cn, axis=1).sum()
completely_blank = total_rows - has_data

print("\n========================================")
print("BATCH PROCESSING (UNTOUCHED ROWS FIRST)")
print("========================================")
print(f"Total rows            : {total_rows}")
print(f"Rows with CNs exists  : {has_data} (will skip)")
print(f"Completely blank rows : {completely_blank} (to process)")
print(f"CSV                   : {CSV_PATH}")
print("========================================")


# ============================================================
# BATCH PROCESSING
# ============================================================

for i in range(total_rows):

    # --------------------------------------------------------
    # Skip if the row already has ANY CN filled
    # --------------------------------------------------------

    if row_has_any_cn(df.iloc[i]):
        print(f"[{i + 1}/{total_rows}] Has data already → skipping")
        continue

    hate_speech = str(df.at[i, "text"]).strip()

    # Safety check for empty input text
    if not hate_speech or hate_speech.lower() == "nan":
        print(f"[{i + 1}/{total_rows}] ⚠️ Empty text → skipping")
        continue

    print(f"\n[{i + 1}/{total_rows}] Processing untouched row...")

    try:

        # ----------------------------------------------------
        # Run existing pipeline
        # ----------------------------------------------------

        result = run_pipeline_quiet(hate_speech)

        if not isinstance(result, dict):
            raise TypeError(
                f"run_pipeline() returned {type(result).__name__}, "
                "expected dict."
            )

        # ----------------------------------------------------
        # Extract counter narratives
        # ----------------------------------------------------

        counter_narratives = result.get(
            "counter_narratives",
            {}
        )

        if not isinstance(counter_narratives, dict):
            raise TypeError(
                "counter_narratives is not a dictionary."
            )

        factual = str(
            counter_narratives.get("Fact-based", "")
        ).strip()

        positive = str(
            counter_narratives.get("Positive Response", "")
        ).strip()

        counter_question = str(
            counter_narratives.get("Counter Question", "")
        ).strip()

        # ----------------------------------------------------
        # Write results into dataframe
        # ----------------------------------------------------

        df.at[i, "factual_cn"] = factual
        df.at[i, "counter_question_cn"] = counter_question
        df.at[i, "positive_cn"] = positive

        # ----------------------------------------------------
        # IMMEDIATELY SAVE PROGRESS
        # ----------------------------------------------------

        save_progress()

        # ----------------------------------------------------
        # Report result
        # ----------------------------------------------------

        if row_is_complete(df.iloc[i]):
            print(
                f"[{i + 1}/{total_rows}] "
                f"✅ Completed and saved"
            )
        else:
            print(
                f"[{i + 1}/{total_rows}] "
                f"⚠️ Saved, but one or more outputs are empty"
            )

    except Exception as e:

        # ----------------------------------------------------
        # Do not destroy previous progress if one row fails.
        # ----------------------------------------------------

        print(
            f"[{i + 1}/{total_rows}] ❌ Failed: "
            f"{type(e).__name__}: {e}"
        )

        # Save all progress accumulated so far.
        try:
            save_progress()
            print(
                f"[{i + 1}/{total_rows}] "
                f"💾 Progress saved"
            )
        except Exception as save_error:
            print(
                f"[{i + 1}/{total_rows}] "
                f"🚨 Could not save progress: "
                f"{type(save_error).__name__}: {save_error}"
            )

    # --------------------------------------------------------
    # Small delay before next row
    # --------------------------------------------------------

    time.sleep(SLEEP_BETWEEN_ROWS)


# ============================================================
# FINAL CHECK
# ============================================================

completed = df.apply(row_is_complete, axis=1).sum()
remaining_blank = total_rows - df.apply(row_has_any_cn, axis=1).sum()

print("\n========================================")
print("BATCH PROCESSING FINISHED")
print("========================================")
print(f"Fully completed rows : {completed}/{total_rows}")
print(f"Completely blank left: {remaining_blank}")
print(f"Saved to             : {CSV_PATH}")
print("========================================")

In [ ]:
#### if any of 3 is missing- run that
import os
import time
import contextlib
import io
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "/content/drive/MyDrive/Hindi_CN_Research/final_cn_output.csv"

# Pause between pipeline runs.
SLEEP_BETWEEN_ROWS = 2


# ============================================================
# LOAD CURRENT CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

required_columns = [
    "text",
    "factual_cn",
    "counter_question_cn",
    "positive_cn",
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print(f"Loaded {len(df)} rows.")
print("Columns:", df.columns.tolist())


# ============================================================
# QUIET VERSION OF EXISTING PIPELINE
# ============================================================

def run_pipeline_quiet(hate_speech):
    """
    Runs the existing pipeline while hiding its intermediate
    stdout output from the notebook.
    """
    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        result = run_pipeline(hate_speech)

    return result


# ============================================================
# HELPER: SAFE NON-EMPTY CHECK
# ============================================================

def is_nonempty(value):
    """
    Correctly handles both empty strings and pandas NaN values.
    """
    return pd.notna(value) and str(value).strip() != ""


# ============================================================
# HELPER: CHECK WHETHER A ROW IS COMPLETE
# ============================================================

def row_is_complete(row):
    return (
        is_nonempty(row["factual_cn"])
        and is_nonempty(row["counter_question_cn"])
        and is_nonempty(row["positive_cn"])
    )


# ============================================================
# HELPER: SAFE CSV SAVE
# ============================================================

def save_progress():
    """
    Save to a temporary file first, then replace the original.
    This reduces the chance of leaving behind a partially-written CSV.
    """
    temp_path = CSV_PATH + ".tmp"

    df.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(temp_path, CSV_PATH)


# ============================================================
# PRE-RUN STATUS
# ============================================================

total_rows = len(df)

already_complete = df.apply(row_is_complete, axis=1).sum()
remaining = total_rows - already_complete

print("\n========================================")
print("BATCH PROCESSING")
print("========================================")
print(f"Total rows       : {total_rows}")
print(f"Already complete : {already_complete}")
print(f"Remaining        : {remaining}")
print(f"CSV              : {CSV_PATH}")
print("========================================")


# ============================================================
# BATCH PROCESSING
# ============================================================

for i in range(total_rows):

    # --------------------------------------------------------
    # Skip rows that were already completed.
    # --------------------------------------------------------

    if row_is_complete(df.iloc[i]):
        print(f"[{i + 1}/{total_rows}] Already complete → skipping")
        continue

    hate_speech = str(df.at[i, "text"]).strip()

    # Safety check for empty input text
    if not hate_speech:
        print(f"[{i + 1}/{total_rows}] ⚠️ Empty text → skipping")
        continue

    print(f"\n[{i + 1}/{total_rows}] Processing...")

    try:

        # ----------------------------------------------------
        # Run existing pipeline
        # ----------------------------------------------------

        result = run_pipeline_quiet(hate_speech)

        if not isinstance(result, dict):
            raise TypeError(
                f"run_pipeline() returned {type(result).__name__}, "
                "expected dict."
            )

        # ----------------------------------------------------
        # Extract counter narratives
        # ----------------------------------------------------

        counter_narratives = result.get(
            "counter_narratives",
            {}
        )

        if not isinstance(counter_narratives, dict):
            raise TypeError(
                "counter_narratives is not a dictionary."
            )

        factual = str(
            counter_narratives.get("Fact-based", "")
        ).strip()

        positive = str(
            counter_narratives.get("Positive Response", "")
        ).strip()

        counter_question = str(
            counter_narratives.get("Counter Question", "")
        ).strip()

        # ----------------------------------------------------
        # Write results into dataframe
        # ----------------------------------------------------

        df.at[i, "factual_cn"] = factual
        df.at[i, "counter_question_cn"] = counter_question
        df.at[i, "positive_cn"] = positive

        # ----------------------------------------------------
        # IMMEDIATELY SAVE PROGRESS
        # ----------------------------------------------------

        save_progress()



        # ----------------------------------------------------
        # Report result
        # ----------------------------------------------------

        if row_is_complete(df.iloc[i]):
            print(
                f"[{i + 1}/{total_rows}] "
                f"✅ Completed and saved"
            )
        else:
            print(
                f"[{i + 1}/{total_rows}] "
                f"⚠️ Saved, but one or more outputs are empty"
            )

    except Exception as e:

        # ----------------------------------------------------
        # Do not destroy previous progress if one row fails.
        # ----------------------------------------------------

        print(
            f"[{i + 1}/{total_rows}] ❌ Failed: "
            f"{type(e).__name__}: {e}"
        )

        # Save all progress accumulated so far.
        try:
            save_progress()
            print(
                f"[{i + 1}/{total_rows}] "
                f"💾 Progress saved"
            )
        except Exception as save_error:
            print(
                f"[{i + 1}/{total_rows}] "
                f"🚨 Could not save progress: "
                f"{type(save_error).__name__}: {save_error}"
            )

    # --------------------------------------------------------
    # Small delay before next row
    # --------------------------------------------------------

    time.sleep(SLEEP_BETWEEN_ROWS)


# ============================================================
# FINAL CHECK
# ============================================================

completed = df.apply(row_is_complete, axis=1).sum()
remaining = total_rows - completed

print("\n========================================")
print("BATCH PROCESSING FINISHED")
print("========================================")

print(f"Completed : {completed}/{total_rows}")
print(f"Remaining : {remaining}")
print(f"Saved to  : {CSV_PATH}")
print("========================================")